# FOLLOW THE MONEY — Monday pre-open, ~10 min, zero config
**1 Tape** (where, with what conviction) → **2 COT** (who: hedge funds / mutual funds / retail — index level) → **3 Insiders** (auto-sweep: who filed, bought, sold, scheduled) → **4 Crowd shorts**.

*Honest limit: per-STOCK attribution of retail/HF/mutual-fund flows doesn't exist publicly — trader-class splits are index-level (COT); stock-level reads come from volume, insiders, and shorts. Run every cell top to bottom; paste outputs back for the vault log.*

In [ ]:
# 1 · THE TAPE — where money moved last week (direction x conviction)
%pip -q install yfinance
import yfinance as yf, pandas as pd, numpy as np, requests
from io import StringIO

SECTORS = {"XLK":"tech","SMH":"semis","XLE":"energy","XLV":"health","XLF":"fins","XLI":"indus",
           "XLP":"staples","XLY":"discret","XLU":"utes","XLB":"materials","XLRE":"REITs",
           "XLC":"comms","GLD":"gold","USO":"crude","TLT":"20y bond","UUP":"dollar","IBIT":"btc"}
rows = []
for t, name in SECTORS.items():
    h = yf.Ticker(t).history(period="4mo")
    if h.empty: continue
    dv = h.Close * h.Volume
    rows.append({"etf": t, "sector": name,
        "1w%": round((h.Close.iloc[-1]/h.Close.iloc[-6]-1)*100, 1),
        "1m%": round((h.Close.iloc[-1]/h.Close.iloc[-22]-1)*100, 1),
        "conviction_x": round(dv.iloc[-5:].mean()/dv.iloc[-65:-5].mean(), 2)})
print(pd.DataFrame(rows).sort_values("1w%", ascending=False).to_string(index=False))
print("\n>1.3x = money moving with intent; ~1.0 = drift; red on >1.3x = money LEAVING\n")

UA = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
html = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers=UA, timeout=30).text
tickers = [str(t).replace(".", "-") for t in pd.read_html(StringIO(html))[0]["Symbol"]]
px = yf.download(tickers, period="4mo", auto_adjust=True, progress=False)
close, vol = px["Close"], px["Volume"]
dv = close * vol
ret1w = (close.iloc[-1]/close.iloc[-6]-1)*100
flow = (dv.iloc[-5:].mean()/dv.iloc[-65:-5].mean()).replace([np.inf], np.nan)
f = pd.DataFrame({"1w%": ret1w.round(1), "flow_x": flow.round(2)}).dropna()
hot = f[f.flow_x > 1.5]
ins, outs = hot[hot["1w%"] > 0], hot[hot["1w%"] < 0]
print(f"SINGLE NAMES — MONEY IN ({len(ins)}):")
print(ins.nlargest(12, "flow_x").to_string() if len(ins) else "  NONE (drift week — no conviction buying anywhere in the S&P 500)")
print(f"\nSINGLE NAMES — MONEY OUT ({len(outs)}):")
print(outs.nlargest(12, "flow_x").to_string() if len(outs) else "  NONE")


In [ ]:
# 2 · WHO MOVED IT — COT by trader class (HFs / mutual funds / retail), index level
# Leveraged Funds = hedge funds | Asset Managers = mutual funds & pensions | Nonreportable = retail proxy
import pandas as pd, requests

def cot(dataset, mkt, prefix, label):
    r = requests.get(f"https://publicreporting.cftc.gov/resource/{dataset}.json",
                     params={"$where": f"contract_market_name='{mkt}'",
                             "$order": "report_date_as_yyyy_mm_dd DESC", "$limit": 8}, timeout=30)
    rows = r.json()
    if not rows: print(f"  {label}: no rows"); return
    d = pd.DataFrame(rows)
    lc = [c for c in d.columns if prefix in c and "long" in c and not any(x in c for x in ("spread","change","pct"))]
    sc = [c for c in d.columns if prefix in c and "short" in c and not any(x in c for x in ("spread","change","pct"))]
    if not (lc and sc): print(f"  {label}: no cols"); return
    d["net"] = d[lc[0]].astype(float) - d[sc[0]].astype(float)
    cur, wk1, wk4 = d.net.iloc[0], d.net.iloc[min(1,len(d)-1)], d.net.iloc[min(4,len(d)-1)]
    print(f"  {label:<22} net {cur:+,.0f} | 1wk delta {cur-wk1:+,.0f} | 4wk delta {cur-wk4:+,.0f}")

for mkt, tag in [("E-MINI S&P 500", "S&P 500"), ("NASDAQ MINI", "NASDAQ")]:
    print(f"{tag}  (report date = Tuesday, released Friday)")
    cot("gpe5-46if", mkt, "lev_money", "hedge funds")
    cot("gpe5-46if", mkt, "asset_mgr", "mutual funds/pensions")
    cot("gpe5-46if", mkt, "nonrept",   "retail (nonreportable)")
    print()
print("CRUDE / GOLD (managed money = the spec class)")
cot("72hh-3qpy", "CRUDE OIL, LIGHT SWEET-WTI", "m_money", "crude specs")
cot("72hh-3qpy", "GOLD", "m_money", "gold specs")
print("\nread: LEVELS are structural; DELTAS are the week's story. Extremes = stored reversal energy.")


In [ ]:
# 3 · INSIDERS — automatic: sweep watchlist, parse whoever filed, buys vs sells vs scheduled
import requests, time, re
import pandas as pd
import xml.etree.ElementTree as ET
from datetime import date, timedelta

H = {"User-Agent": "Jake research 7bm4q6x5sm@privaterelay.appleid.com"}
WATCH = sorted(set(
    ["MU","LRCX","AMAT","NVDA","AMD","AVGO","KLAC","TER","ON","MRVL","MPWR","SMCI","WDC","STX","SNDK"] +
    ["PG","GILD","CME","GD","HIG","EA","FFIV","ULTA","GEHC","CBOE","UHS","WRB","CB","HII","CI"] +
    ["VRRM","GTM","DUOL","INSP","EPAM","FRPT","BLKB","AMPH","GPK","ADMA","UPWK","WEN"] +
    ["CLF","VICR","VSH","ENS","VECO","AXTI","POWL","STRL","FIX","TTMI"] +
    ["LLY","NOW","ORCL"]))
SINCE = (date.today() - timedelta(days=7)).isoformat()
CODES = {"P":"BUY","S":"SELL","A":"grant","M":"exercise","F":"tax-wh","G":"gift"}

cm = requests.get("https://www.sec.gov/files/company_tickers.json", headers=H, timeout=30).json()
cik = {v["ticker"]: str(v["cik_str"]).zfill(10) for v in cm.values()}

def txt(el, path):
    e = el.find(path)
    return e.text.strip() if e is not None and e.text else None

rows = []
for t in WATCH:
    c = cik.get(t)
    if not c: continue
    try:
        sub = requests.get(f"https://data.sec.gov/submissions/CIK{c}.json", headers=H, timeout=30).json()
    except Exception: continue
    time.sleep(0.12)
    rec = sub["filings"]["recent"]
    f4s = [(rec["accessionNumber"][i], rec["filingDate"][i], rec["primaryDocument"][i])
           for i in range(len(rec["form"])) if rec["form"][i] == "4" and rec["filingDate"][i] >= SINCE]
    for acc, fdate, pdoc in f4s:
        url = f"https://www.sec.gov/Archives/edgar/data/{int(c)}/{acc.replace('-','')}/{pdoc.split('/')[-1]}"
        try:
            x = requests.get(url, headers=H, timeout=30).text
            time.sleep(0.12)
            root = ET.fromstring(re.sub(r"<\?xml[^>]*\?>", "", x, count=1))
        except Exception: continue
        owner = txt(root, ".//reportingOwner/reportingOwnerId/rptOwnerName") or "?"
        rel = root.find(".//reportingOwner/reportingOwnerRelationship")
        title = (txt(rel, "officerTitle") or ("Director" if txt(rel, "isDirector") in ("1","true") else "?"))
        plan = txt(root, ".//aff10b5One") in ("1", "true")
        for tr in root.findall(".//nonDerivativeTransaction"):
            code_ = txt(tr, "transactionCoding/transactionCode")
            sh, pr = txt(tr, "transactionAmounts/transactionShares/value"), txt(tr, "transactionAmounts/transactionPricePerShare/value")
            if not (code_ and sh): continue
            rows.append({"ticker": t, "filed": fdate, "owner": owner[:24], "title": title[:20],
                         "type": CODES.get(code_, code_), "value_$": round(float(sh)*float(pr or 0)),
                         "10b5-1": "sched" if plan else ""})
    print(".", end="")

df = pd.DataFrame(rows)
if len(df):
    real = df[df.type.isin(["BUY","SELL"])]
    print("\n\n=== DISCRETIONARY + SCHEDULED BUYS/SELLS (grants/noise filtered) ===")
    print(real.sort_values(["ticker","filed"]).to_string(index=False))
    print("\n=== VERDICTS ===")
    for t in real.ticker.unique():
        d = real[(real.ticker == t) & (real["10b5-1"] == "")]
        buys, sells = d[d.type == "BUY"], d[d.type == "SELL"]
        sched = real[(real.ticker == t) & (real["10b5-1"] == "sched") & (real.type == "SELL")]["value_$"].sum()
        nb = buys.owner.nunique()
        tag = " <<< CLUSTER BUY" if nb >= 3 else ""
        print(f"{t}: disc buys ${buys['value_$'].sum():,} ({nb} buyers{tag}) | disc sells ${sells['value_$'].sum():,} | sched sells ${sched:,}")
else:
    print("\nno watchlist Form 4s this week")


In [ ]:
# 4 · CROWD SHORTS — short interest on the watchlist (bi-monthly data; MoM change = the signal)
import yfinance as yf
for t in ["MU","NVDA","SMCI","ORCL","GPK","FRPT","DUOL","UPWK","GTM","VECO","AXTI","ENS","VICR","CLF"]:
    try:
        i = yf.Ticker(t).info
        spf, ss, ssp = i.get("shortPercentOfFloat"), i.get("sharesShort"), i.get("sharesShortPriorMonth")
        d = f"{(ss/ssp-1)*100:+.0f}% MoM" if ss and ssp else "n/a"
        print(f"{t}: {spf*100:.1f}% of float | {d}" if spf else f"{t}: n/a")
    except Exception: print(f"{t}: err")


*Key: buys are choices, sells are often calendars · deltas > levels · crowding = stored reversal energy · insiders lead by months, tape is coincident, 13Fs are history.*